# api_model_difference_intent_v1

目标：验证 `src/api` 的 paper intent 能读取 dashboard 的 `model_difference_explanations`。当 top model 被模型差异解释标记为 loser 且 decision 为 `block_or_rework_loser` 时，intent 必须降到 `min_position`，并把差异上下文写入 ledger，避免 paper/sim 链路忽略 analysis 风险信号。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.api import PaperBrokerAdapter, build_target_position_intent_from_dashboard, execute_paper_intent, write_intent_ledger

EXPERIMENT_DIR = REPO_ROOT / "research/api_model_difference_intent_v1"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, payload: dict | list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

dashboard_payload = {
    "schema_version": "dashboard.model_matrix.v1",
    "artifact_id": "api_model_difference_intent_v1",
    "top_models": [
        {
            "rank": 1,
            "model": "random_forest_clf__all_features",
            "base_model": "random_forest_clf",
            "feature_set": "all_features",
            "avg_position": 0.74,
            "stability_score": 54.2,
            "max_drawdown_pct": 21.6,
        }
    ],
    "model_difference_explanations": [
        {
            "schema_version": "analysis.model_difference_explanation.v1",
            "priority_rank": 1,
            "segment_type": "window_regime",
            "segment": "stress_window::stress",
            "baseline_model": "ridge_mse__all_features",
            "winner_model": "ridge_mse__all_features",
            "loser_model": "random_forest_clf__all_features",
            "relative_return_pct_spread": 6.5,
            "explanation_signal": "baseline_defends",
            "decision_hint": "block_or_rework_loser",
            "review_note": "中文 review：top model 在 stress regime 被 ridge_mse 明显压制，paper intent 应进入最小仓位。",
        }
    ],
}
write_json(EXPERIMENT_DIR / "dashboard_export.json", dashboard_payload)

intent = build_target_position_intent_from_dashboard(
    dashboard_payload,
    instrument="SOXL",
    min_position=0.1,
    max_position=0.75,
)
result = execute_paper_intent(intent, broker=PaperBrokerAdapter(cash=10000.0), price=100.0)
ledger_path = write_intent_ledger(EXPERIMENT_DIR / "intent_ledger.json", intent=intent, result=result)

intent.target_position, result.snapshot.shares, ledger_path.name


(0.1, 10, 'intent_ledger.json')

In [2]:
summary = {
    "status": "completed",
    "schema_version": "research.api_model_difference_intent_v1",
    "source_artifact": dashboard_payload["artifact_id"],
    "source_model": intent.source_model,
    "raw_avg_position": intent.metadata["raw_avg_position"],
    "target_position": intent.target_position,
    "execution_gate": intent.metadata["execution_gate"],
    "model_difference_top_signal": intent.metadata["model_difference_top_signal"],
    "model_difference_top_decision_hint": intent.metadata["model_difference_top_decision_hint"],
    "model_difference_top_segment": intent.metadata["model_difference_top_segment"],
    "model_difference_top_loser_model": intent.metadata["model_difference_top_loser_model"],
    "model_difference_top_winner_model": intent.metadata["model_difference_top_winner_model"],
    "model_difference_top_spread": intent.metadata["model_difference_top_spread"],
    "paper_result_status": result.status,
    "paper_snapshot_shares": result.snapshot.shares,
    "sidecars": {
        "dashboard_export": str(EXPERIMENT_DIR / "dashboard_export.json"),
        "intent_ledger": str(EXPERIMENT_DIR / "intent_ledger.json"),
        "analysis_summary": str(EXPERIMENT_DIR / "analysis_summary.json"),
    },
}
write_json(EXPERIMENT_DIR / "analysis_summary.json", summary)

conclusion = """# 中文结论

- 做了什么：验证 `src/api` 从 dashboard 的 `model_difference_explanations` 读取模型差异解释，并写入 paper intent ledger。
- 核心指标：top model `random_forest_clf__all_features` 的原始 `avg_position=0.74`，因为在 `stress_window::stress` 被标记为 loser 且 decision 为 `block_or_rework_loser`，最终 `target_position=0.1`。
- regime 差异：stress regime 中 winner 是 `ridge_mse__all_features`，loser 是 `random_forest_clf__all_features`，相对收益 spread 为 `6.5`。
- 是否晋升：这是 API paper/sim 风控上下文能力验证，可随框架进入 production；不代表交易策略晋升。
- 风险：本实验使用 mock dashboard payload，只验证 contract 传递和 paper adapter 行为，不替代真实回测。
- 下一步：真实 dashboard export 进入 paper execution schedule 后，ledger 可用于审计为什么 target position 被压低。"""

(EXPERIMENT_DIR / "analysis_conclusion.md").write_text(conclusion, encoding="utf-8")
summary


{'status': 'completed',
 'schema_version': 'research.api_model_difference_intent_v1',
 'source_artifact': 'api_model_difference_intent_v1',
 'source_model': 'random_forest_clf__all_features',
 'raw_avg_position': 0.74,
 'target_position': 0.1,
 'execution_gate': 'blocked_to_min_position',
 'model_difference_top_signal': 'baseline_defends',
 'model_difference_top_decision_hint': 'block_or_rework_loser',
 'model_difference_top_segment': 'stress_window::stress',
 'model_difference_top_loser_model': 'random_forest_clf__all_features',
 'model_difference_top_winner_model': 'ridge_mse__all_features',
 'model_difference_top_spread': 6.5,
 'paper_result_status': 'paper_executed',
 'paper_snapshot_shares': 10,
 'sidecars': {'dashboard_export': '/Users/rziiiii/dev/workspace/research/api_model_difference_intent_v1/dashboard_export.json',
  'intent_ledger': '/Users/rziiiii/dev/workspace/research/api_model_difference_intent_v1/intent_ledger.json',
  'analysis_summary': '/Users/rziiiii/dev/workspace/

# 中文结论

- 做了什么：验证 `src/api` 的 paper intent 会消费 dashboard `model_difference_explanations`，并把模型差异上下文写入 intent metadata 和 ledger。
- 核心指标：当 top model 是 loser 且 decision 为 `block_or_rework_loser`，`avg_position=0.74` 被降到 `target_position=0.1`。
- regime 差异：`stress_window::stress` 中 `ridge_mse__all_features` 胜过 `random_forest_clf__all_features`，spread 为 `6.5`。
- 是否晋升：该 notebook 只验证 API paper/sim 风控链路，可随框架能力进入 production；不是策略晋升。
- 风险：使用 mock dashboard payload，不证明模型真实表现。
- 下一步：把真实 dashboard export 产生的差异解释纳入 scheduled paper execution 审计。